In [ ]:
"""
FILE 7: GRAPHS, SAFE ITERATION, DEDUPLICATION, COMMON IDIOMS, PERFORMANCE
============================================================================
Yeh last file hai - isme wo chote chote patterns hain jo DSA solve karte
waqt baar baar kaam aate hain, plus performance ki common galtiyan.
"""

from collections import defaultdict
from heapq import heappush, heappop
import copy


# ===============================================================
# 1) GRAPH REPRESENTATION (Python mein built-in graph class nahi hoti)
# ===============================================================

# APPROACH 1: defaultdict(list) - jab node IDs kuch bhi ho sakte hain (string, bade numbers, etc)
edges = [(0, 1), (1, 2)]
graph_dd = defaultdict(list)
for u, v in edges:
    graph_dd[u].append(v)
    graph_dd[v].append(u)   # undirected graph. Directed ho to yeh line hata do

for neighbor in graph_dd[0]:
    pass   # missing node ke liye bhi khaali list milegi, error nahi aayega

# APPROACH 2: list of lists - jab nodes 0 se n-1 tak fixed ho (zyada memory-efficient)
n = 5
graph_list = [[] for _ in range(n)]
for u, v in edges:
    graph_list[u].append(v)
    graph_list[v].append(u)

# WEIGHTED GRAPH - (neighbor, weight) tuple store karo:
weighted_edges = [(0, 1, 4), (1, 2, 7)]
graph_w = defaultdict(list)
for u, v, w in weighted_edges:
    graph_w[u].append((v, w))
    graph_w[v].append((u, w))

# TABLE (kab konsa approach use karein):
#   defaultdict(list)      -> node IDs bade/sparse/non-numeric hain
#   [[] for _ in range(n)]  -> nodes 0 se n-1 tak hain, FAST index access chahiye

# DIJKSTRA PATTERN (heap use karke shortest path):
def dijkstra(graph, start, n):
    dist = [float('inf')] * n
    dist[start] = 0
    pq = [(0, start)]   # (distance, node)

    while pq:
        d, node = heappop(pq)
        if d > dist[node]:
            continue   # PURANI outdated entry hai, skip karo
        for neighbor, weight in graph[node]:
            new_dist = d + weight
            if new_dist < dist[neighbor]:
                dist[neighbor] = new_dist
                heappush(pq, (new_dist, neighbor))

    return dist


# ===============================================================
# 2) LOOP KE ANDAR COLLECTION MODIFY KARNA - SAWDHAAN!
# ===============================================================

# --- DICT ---
# GALAT: dict ka size loop ke beech mein change karna RuntimeError deta hai
d = {'a': 1, 'b': 2, 'c': 3}
# for key in d:
#     if d[key] < 2:
#         del d[key]     # RuntimeError: dictionary changed size during iteration

# SAHI 1: keys ki COPY banao pehle
for key in list(d.keys()):
    if d[key] < 2:
        del d[key]

# SAHI 2: naya dict comprehension se banao
d2 = {'a': 1, 'b': 2, 'c': 3}
d2 = {k: v for k, v in d2.items() if v >= 2}

# --- SET --- (same problem)
s = {1, 2, 3, 4, 5}
# for x in s:
#     if x % 2 == 0:
#         s.remove(x)    # RuntimeError!

for x in list(s):    # COPY banao pehle
    if x % 2 == 0:
        s.discard(x)

s = {x for x in {1, 2, 3, 4, 5} if x % 2 != 0}   # comprehension approach

# --- LIST --- (error nahi aata, par SILENT BUG aata hai - yeh zyada khatarnak hai!)
lst = [1, 2, 3, 4, 5]
# GALAT: index shift ho jaate hain, kuch elements skip ho jaate hain
# for i in range(len(lst)):
#     if lst[i] % 2 == 0:
#         lst.pop(i)     # index 1 hatane ke baad, index 2 wala ab index 1 pe aa gaya!

# SAHI 1: PEECHE se (backward) traverse karo
lst = [1, 2, 3, 4, 5]
for i in range(len(lst) - 1, -1, -1):
    if lst[i] % 2 == 0:
        lst.pop(i)

# SAHI 2 (SABSE PYTHONIC): naya list banao comprehension se
lst2 = [1, 2, 3, 4, 5]
lst2 = [x for x in lst2 if x % 2 != 0]

# PRACTICAL TIP: zyadatar DSA problems mein "iterate karte hue remove karna"
# zaroori nahi hota - seedha NAYA collection bana lo desired elements ke saath.


# ===============================================================
# 3) DEDUPLICATION PATTERNS (duplicate results hatana)
# ===============================================================
# Jaise 3Sum, 4Sum, permutations-with-duplicates jaise problems mein zaroori

# APPROACH A: SORT KARKE SKIP KARO (INTERVIEW MEIN PREFERRED)
def dedup_sort_skip(nums):
    nums.sort()
    result = []
    for i in range(len(nums)):
        if i > 0 and nums[i] == nums[i - 1]:
            continue   # SAME value dobara aayi hai - skip karo
        result.append(nums[i])
    return result
# FAYDA: extra space nahi lagta (sort ke alawa), result already sorted milta hai

# APPROACH B: SET-BASED DEDUPLICATION (simpler, par thoda zyada memory)
def dedup_using_set(candidates):
    result_set = set()
    for a, b, c in candidates:
        result_set.add(tuple(sorted([a, b, c])))   # list hashable nahi, TUPLE banao
    return [list(t) for t in result_set]   # wapas list of lists mein convert karo

# Interview mein "sort and skip" zyada pasand kiya jaata hai kyunki hashing ka
# overhead nahi hota aur code cleaner hota hai. Par set-wala tarika implement
# karna easier hai jab sort-skip complicated lage.


# ===============================================================
# 4) COMMON IDIOMS (chote chote helpful tricks)
# ===============================================================

# SENTINEL VALUES (initial min/max set karne ke liye):
min_val = float('inf')     # koi bhi number isse chota hoga -> minimum dhundhne ke liye
max_val = float('-inf')      # koi bhi number isse bada hoga -> maximum dhundhne ke liye
from math import inf
min_val2 = inf
max_val2 = -inf

# MATH UTILITIES:
print(max(3, 7))                  # built-in maximum
print(min(3, 7))                   # built-in minimum
print(abs(-5))                      # absolute value
print(pow(2, 10))                    # power (2**10 jaisa hi)
print(pow(2, 10, 1000))               # MODULAR exponentiation - built-in & fast!
print(divmod(7, 3))                    # (2, 1) -> (quotient, remainder) ek saath

from math import isqrt, gcd, lcm, ceil, floor, log2
print(isqrt(17))     # 4 -> integer square root (Python 3.8+)
print(gcd(12, 8))      # 4 -> greatest common divisor
print(lcm(4, 6))         # 12 -> least common multiple (Python 3.9+)
print(ceil(7 / 2))         # 4
print(floor(7 / 2))          # 3

# CEILING DIVISION - bina math import kiye (float precision issues se bachne ke liye):
a, b = 7, 2
ceiling_div = (a + b - 1) // b   # yeh math.ceil(a/b) se zyada SAFE hai bade integers ke liye

# DEEP COPY vs SHALLOW COPY:
lst_orig = [1, 2, 3]
copy1 = lst_orig[:]           # shallow copy (slicing)
copy2 = list(lst_orig)         # shallow copy (constructor)
copy3 = lst_orig.copy()         # shallow copy (method)

# 2D LIST ke liye SIRF shallow copy KAAFI NAHI hai!
matrix = [[1, 2], [3, 4]]
matrix_copy = [row[:] for row in matrix]   # HAR row ko alag se copy karna padega

# Deeply NESTED structures ke liye:
nested = {"a": [1, [2, 3]]}
deep = copy.deepcopy(nested)   # sab kuch fully independent copy ho jaata hai

# TYPE CONVERSIONS (yeh cheat sheet yaad rakho):
print(tuple([1, 2, 3]))         # list -> tuple (hashable banane ke liye)
print(list((1, 2, 3)))            # tuple -> list (mutable banane ke liye)
print(list("hello"))                # string -> list of chars
print("".join(['h', 'i']))            # list of chars -> string
print(int("42"))                        # string -> int
print(str(42))                            # int -> string
print(int("1010", 2))                       # binary string -> int (base 2)
print(bin(10))                                # int -> binary string
print(set([1, 1, 2]))                           # duplicates hatao
print(sorted({3, 1, 2}))                          # set -> sorted list

# INFINITY AUR CHAINED COMPARISONS:
print(float('inf') > 10**100)   # True - infinity kisi bhi number se bada hota hai

dist = [float('inf')] * 5
dist[0] = 0   # start node ka distance 0, baaki sab abhi infinity hain

x, n = 5, 10
if 0 <= x < n:            # yeh (x >= 0 and x < n) jaisa hi hai - PYTHONIC tarika
    pass

a_, b_, c_ = 1, 2, 3
if a_ <= b_ <= c_:          # chained comparison - (a<=b and b<=c) jaisa
    pass


# ===============================================================
# 5) PERFORMANCE TIPS (TLE - Time Limit Exceeded se bachne ke liye)
# ===============================================================
# SLOW               ->  FAST                    ->  KYUN
# list.pop(0)             deque.popleft()               O(n) vs O(1) [BFS queue]
# s += char in loop        "".join(parts)                 O(n^2) vs O(n) [string build]
# x in list                  x in set                       O(n) vs O(1) [membership test]
# list.insert(0, x)           deque.appendleft(x)             O(n) vs O(1) [insert at front]
# insort(list, x)               SortedList.add(x)                O(n) vs O(log n) [sorted insert]
# copy.deepcopy()                 [row[:] for row in matrix]        General vs specific (fast)

# TIPS YAAD RAKHNE KE LIYE:
#   1. SAHI data structure choose karo - set (lookup ke liye), deque (BFS ke liye),
#      heapq (priority queue ke liye)
#   2. FALTU copies avoid karo - slicing naya list banati hai, jahan ho sake index use karo
#   3. @cache use karo memoization ke liye - manual dict se aksar fast hota hai
#   4. list comprehension .append() waale loop se FAST hota hai
#   5. LOCAL variables globals se FAST hote hain - agar function baar baar
#      global use kar raha hai, to use pehle local variable mein copy kar lo
#   6. str += char loop mein MAT karo, "".join() use karo
#   7. Deep recursion ke liye sys.setrecursionlimit() use karo (iteration mein
#      convert karna sirf tab jab depth SACH MEIN bahut zyada ho)